# project_02_mpnn_optimization — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [ ]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [ ]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

In [ ]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [ ]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [ ]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — the MPNN trade-off + the design/recapitulation loop

**Standard slot:** *define & explore.* **For Project 02 this means:** pin down the
recovery ↔ foldability ↔ expressibility trade-off and the metrics that measure it, then stand up
`mpnn_tools` plus a recapitulation step and run them on **one** backbone as your "hello-world" (D0):
design sequences, compute sequence recovery, and a (mock) recapitulation scRMSD.

Run `00_setup.ipynb` first in this session.

## The metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| Sequence recovery | 0–1 | fraction of positions matching a native/reference | design *quality* (high recovery can just mean low diversity) |
| Recapitulation scRMSD | Å | designed→predicted-vs-input-backbone Cα-RMSD (<2 Å self-consistent) | that the protein folds, is stable, or expresses |
| pLDDT (mean) | 0–100 | local confidence of the recapitulation | thermostability / expression |
| Per-position entropy | bits | sequence *diversity* across samples | correctness |
| Net charge (pH 7.4) | ± | a solubility *proxy* (charge extremes hurt solubility) | a measured solubility |
| Hydrophobic-patch fraction | 0–1 | aggregation-risk *proxy* (exposed hydrophobic runs) | a measured aggregation rate |
| `camsol_like` | unitless | a CamSol-**style** heuristic (higher ≈ more soluble) | **real CamSol**; not an expression result |

Write your own one-paragraph definitions in `D0` — including the "does not mean" column, which is
where wasted synthesis budget comes from (e.g., treating a proxy score as an expression prediction).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Stand up `mpnn_tools`

`scripts/mpnn_tools.py` exposes `run_mpnn(backbone, temperature, noise, n_seqs, tool)` plus the
metric helpers. The real backend shells out to ProteinMPNN (see the TODO in `_real_mpnn`); a
deterministic **mock** backend lets you build and test the sweep, recovery, entropy, and solubility
logic first. **Never report mock sequences or proxy scores as real results.**

In [ ]:
from mpnn_tools import (run_mpnn, sequence_recovery, shannon_entropy,
                        net_charge, hydrophobic_fraction, camsol_like)

# Design 8 sequences for ONE backbone with the mock backend (no GPU, no MPNN install).
designs = run_mpnn("demo_backbone_01", temperature=0.2, noise=0.1, n_seqs=8, tool="mock")
print(f"designed {len(designs)} sequences for backbone {designs[0].backbone!r}")
print("first sequence:", designs[0].sequence)

## Hello-world: recovery + a mock recapitulation for one backbone

Sequence recovery needs a *native* reference. For a de novo backbone there is no single native —
use the mock backend's own consensus as a stand-in here so the plumbing runs; with the real backend
you compute recovery vs the backbone's reference sequence (natural references) or report inter-design
recovery (de novo). Then we *recapitulate*: a real run predicts each sequence's structure (ESMFold/AF2)
and measures Cα-RMSD back to the input backbone. Here a small **mock predict** stands in so the loop
executes anywhere.

In [ ]:
import hashlib

# A stand-in "native" so recovery is meaningful in the dry run: re-derive the mock consensus.
# (With the real backend, recovery is vs the backbone's reference sequence — see MANUAL §2.)
ref = run_mpnn("demo_backbone_01", temperature=0.01, noise=0.0, n_seqs=1, tool="mock")[0].sequence

def mock_recapitulate(sequence):
    """Deterministic stand-in for ESMFold/AF2 recapitulation. NOT a real prediction.
    Returns (scrmsd_A, plddt). Replace with a real predict() on Colab (see scripts/predict.py
    pattern in Project 01)."""
    h = int(hashlib.sha256(sequence.encode()).hexdigest(), 16)
    scrmsd = round(0.8 + (h % 350) / 100.0, 2)   # 0.8–4.3 Å, synthetic
    plddt = round(60 + (h % 40), 1)              # 60–99, synthetic
    return scrmsd, plddt

for d in designs:
    d.recovery = sequence_recovery(ref, d.sequence)
    d.scrmsd, d.plddt = mock_recapitulate(d.sequence)

print("EXAMPLE_DATA (mock backend) — one backbone, 8 sequences:")
for d in designs:
    print(f"  seq {d.seq_index}: recovery={d.recovery:.2f}  scrmsd={d.scrmsd} A  "
          f"plddt={d.plddt}  net_charge={d.net_charge}  hp_frac={d.hydrophobic_fraction}  "
          f"camsol_like={d.camsol_like}")

### Diversity at one glance

Per-position Shannon entropy across the 8 sequences — the diversity axis you will sweep against
temperature. Near-zero means MPNN is confident (low temperature); higher means it samples broadly.

In [ ]:
ent = shannon_entropy([d.sequence for d in designs])
print(f"mean per-position entropy = {ent['mean']:.3f} bits over {ent['length']} positions")
print("Try re-running run_mpnn(...) with temperature=0.1 vs 0.5 and watch entropy move.")

### Switch to the real backend (on Colab)

Once you have cloned ProteinMPNN (pin the commit) and installed ESMFold via `00_setup`'s
`install_esmfold()`, change `tool="mock"` to `tool="proteinmpnn"` in `run_mpnn(...)` and replace
`mock_recapitulate` with a real `predict(sequence, tool="esmfold")` → scRMSD. Record runtime +
versions + the pinned commit in `LOG.md`. **MPNN is seconds; the predictor is your compute cost.**

In [ ]:
# Uncomment on Colab after wiring up ProteinMPNN + ESMFold:
# real = run_mpnn("data/inputs/backbone_001.pdb", temperature=0.2, noise=0.1, n_seqs=8,
#                 tool="proteinmpnn")
# from predict import predict   # Project 01-style wrapper; compute Ca-RMSD vs the input backbone
# ...
print("Ready — flip tool='mock' to 'proteinmpnn' and wire in a real predict() on Colab.")

## D0 checklist
- [ ] One-paragraph definition of each metric **with** its "does not mean" note (esp. proxy ≠ expression).
- [ ] One backbone designed; sequence recovery + a (mock, then real) recapitulation scRMSD printed.
- [ ] Per-position entropy printed; you have watched it move with temperature.
- [ ] `LOG.md` entry: tool version + pinned ProteinMPNN commit, GPU, runtime, seed.

**Next:** `02_generate.ipynb` — the systematic sweep over the grid.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — the systematic MPNN settings sweep

**Standard slot:** *design campaign.* **For Project 02 this means:** the campaign is a **parameter
sweep**, not novel backbone generation. You design across the grid
temperature {0.1, 0.2, 0.3, 0.5} × backbone noise {0.0, 0.1, 0.2} × seqs/backbone {8, 16, 48} for
each of your 20–30 backbones — **hundreds of sequences** — and write them to
`results/sequences.csv` (D2). The `mock` backend makes this run anywhere; the real ProteinMPNN
command is shown so you can reproduce it on Colab.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (tools change)

ProteinMPNN/LigandMPNN are installed from source and **move** — pin a commit and verify the repo
still exists before a campaign. This cell HTTP-checks the pinned upstream URLs (it is allowed to
fail offline; on Colab it confirms the repos are reachable).

In [ ]:
import requests

# Pinned upstreams (verify + pin a COMMIT in your LOG.md before the course; these change):
#   ProteinMPNN  https://github.com/dauparas/ProteinMPNN   (pin e.g. a commit hash)
#   LigandMPNN   https://github.com/dauparas/LigandMPNN    (pin e.g. a commit hash)
UPSTREAMS = {
    "ProteinMPNN": "https://github.com/dauparas/ProteinMPNN",
    "LigandMPNN":  "https://github.com/dauparas/LigandMPNN",
}
for name, url in UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=10)
        print(f"{name:12s} {url}  -> HTTP {r.status_code}")
    except Exception as e:
        print(f"{name:12s} {url}  -> could not reach ({e}); fine offline, re-check on Colab")
print("\nReminder: pin a COMMIT (not just the repo) and log it — APIs/flags drift between commits.")

## 1 · The sweep grid

Sweep every (temperature, noise, seqs) cell across all backbones. With 20–30 backbones this is
hundreds of sequences. The `mock` backend is deterministic so the loop is reproducible; switch
`TOOL` to `"proteinmpnn"` on Colab.

In [ ]:
import itertools, pandas as pd
from mpnn_tools import run_mpnn

TEMPS  = [0.1, 0.2, 0.3, 0.5]
NOISES = [0.0, 0.1, 0.2]
NSEQS  = [8, 16, 48]
TOOL   = "mock"     # -> "proteinmpnn" on Colab (pin the commit)

# A small stand-in backbone set so the notebook runs end-to-end. REPLACE with your 20-30
# de novo backbones (Project 03 outputs / public design set) dropped into data/inputs/.
BACKBONES = [f"demo_backbone_{i:02d}" for i in range(1, 6)]   # 5 for the dry run; use 20-30 for real
print(f"backbones: {len(BACKBONES)}   grid cells: {len(TEMPS)*len(NOISES)*len(NSEQS)}")

## 2 · Run the sweep → `results/sequences.csv`

Note the **compute budget**: MPNN itself is seconds per backbone (T4 or even CPU). The cost comes
later in notebook 03/04 when you *recapitulate* each sequence with ESMFold/AF2 — so generate broadly
here, then triage. Every row carries its full setting.

In [ ]:
rows = []
for bb in BACKBONES:
    for temp, noise, nseq in itertools.product(TEMPS, NOISES, NSEQS):
        for d in run_mpnn(bb, temperature=temp, noise=noise, n_seqs=nseq, tool=TOOL):
            rows.append(d.as_row())

seqs = pd.DataFrame(rows)
# recapitulation columns (scrmsd, plddt) are filled in nb 03/04 after prediction.
seqs.to_csv("results/sequences.csv", index=False)
print("wrote results/sequences.csv", seqs.shape)
print(seqs[["backbone", "temperature", "noise", "n_seqs", "seq_index",
            "net_charge", "hydrophobic_fraction", "camsol_like"]].head())

### Note on scale (real run)

With 5 demo backbones you already have hundreds of rows because of the seqs/backbone axis. For the
real D2, use 20–30 backbones and the `proteinmpnn` backend. Budget recapitulation, not generation:
recapitulate with ESMFold first (seconds/seq), reserve AF2 full-MSA for survivors, batch overnight.
This is the genuine free-tier bottleneck — see `MANUAL.md §1/§6`.

## 3 · The real ProteinMPNN command (for reproducibility)

The mock backend stands in for this documented call. Pin the commit; log the exact flags.

In [ ]:
REAL_CMD = r"""
# Clone + pin a commit first:
#   git clone https://github.com/dauparas/ProteinMPNN && cd ProteinMPNN && git checkout <COMMIT>
# Then, per backbone × setting:
python ProteinMPNN/protein_mpnn_run.py \
    --pdb_path data/inputs/backbone_001.pdb \
    --out_folder results/mpnn/backbone_001 \
    --num_seq_per_target 16 \
    --sampling_temp "0.2" \
    --backbone_noise "0.1" \
    --seed 37
# Parse results/mpnn/backbone_001/seqs/*.fa into the same schema as results/sequences.csv.
"""
print(REAL_CMD)

## D2 checklist
- [ ] Version-verify cell run; ProteinMPNN/LigandMPNN commit pinned in `LOG.md`.
- [ ] `results/sequences.csv`: full grid × 20–30 backbones, every setting on every row (hundreds of seqs).
- [ ] Design log (settings, seeds, tool versions, runtimes) in `LOG.md`.
- [ ] Compute-budget note: MPNN seconds vs recapitulation cost; your triage plan.
- [ ] 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the shared filter on the swept sequences.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — run the shared multi-layer filter

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 02** you build `fp.Design` objects (`design_type="monomer"`) from each
swept sequence's recapitulation scRMSD/pLDDT + solubility proxies, run the pipeline, and report
survival (D3 part 1).

Run `00`–`02` first so `results/sequences.csv` exists.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — the SAME filter every project uses. We run it in
`design_type="monomer"` mode.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("DEFAULT_CUTOFFS['monomer']:", fp.DEFAULT_CUTOFFS["monomer"])

## Recapitulate the swept sequences (mock here; ESMFold/AF2 on Colab)

`results/sequences.csv` from nb 02 has the sequences but not yet their recapitulation. Fill scRMSD +
pLDDT. Here a deterministic **mock predict** stands in so the filter runs anywhere; on Colab replace
it with a real `predict(seq, tool="esmfold")` (triage) → scRMSD vs the input backbone.

In [ ]:
import hashlib
seqs = pd.read_csv("results/sequences.csv")

def mock_recapitulate(sequence):
    """Deterministic stand-in for ESMFold/AF2. NOT real. EXAMPLE_DATA only."""
    h = int(hashlib.sha256(str(sequence).encode()).hexdigest(), 16)
    return round(0.8 + (h % 350) / 100.0, 2), round(60 + (h % 40), 1)

scr, pl = zip(*[mock_recapitulate(s) for s in seqs["sequence"]])
seqs["scrmsd"], seqs["plddt"] = scr, pl
seqs.to_csv("results/sequences.csv", index=False)   # persist the recapitulation columns
print("recapitulated (mock):", seqs.shape, "| scRMSD range",
      round(seqs.scrmsd.min(),2), "-", round(seqs.scrmsd.max(),2))

## Build `fp.Design` objects (monomer)

Map each swept sequence onto a `fp.Design`: recapitulation `scrmsd`/`plddt`, the solubility proxy as
`solubility` (the filter's physics layer reads it), and stash the setting + proxies in `extra` so we
can group by setting in notebook 04. No `pae_interaction` — this is a monomer.

In [ ]:
designs = []
for _, r in seqs.iterrows():
    designs.append(fp.Design(
        design_id=f"{r['backbone']}|T{r['temperature']}|N{r['noise']}|S{r['n_seqs']}|i{r['seq_index']}",
        sequence=str(r["sequence"]),
        design_type="monomer",
        scrmsd=float(r["scrmsd"]),
        plddt=float(r["plddt"]),
        scrmsd_orthogonal=float(r["scrmsd"]),    # mock: orthogonal == primary; use ESMFold-vs-AF2 for real
        solubility=float(r["camsol_like"]),       # CamSol-STYLE heuristic, NOT real CamSol
        extra=dict(temperature=r["temperature"], noise=r["noise"], n_seqs=r["n_seqs"],
                   net_charge=r["net_charge"], hydrophobic_fraction=r["hydrophobic_fraction"]),
    ))
print(len(designs), "monomer Design objects built")

## Run the pipeline (`design_type="monomer"`)

`run_pipeline` applies the layers in order and returns a ranked DataFrame; `report` prints the
hit-rate accounting and the survival-at-each-layer figure. Monomer cutoffs: scRMSD < 2 Å,
pLDDT > 85 (solubility is a soft physics-layer signal — see `MANUAL.md §4`).

In [ ]:
df_ranked = fp.run_pipeline(designs, design_type="monomer", use_layers=(1, 2, 3))
df_ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(df_ranked, top_n=10, save_prefix="results/proj02")
top

## Survival by setting (honest accounting)

The headline of Project 02 is *which settings survive*. Recover the setting from each `design_id`
and report the per-setting survivor count — the seed of the Pareto analysis in notebook 04.

In [ ]:
import pandas as pd
r = df_ranked.copy()
parts = r["design_id"].str.split("|", expand=True)
r["temperature"] = parts[1].str[1:].astype(float)
r["noise"] = parts[2].str[1:].astype(float)
r["n_seqs"] = parts[3].str[1:].astype(int)
survivors = r[r["layers_passed"] >= 1]
print("survivors (passed L1 self-consistency) by temperature × noise:")
print(survivors.groupby(["temperature", "noise"]).size().unstack(fill_value=0))

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module (not a one-off script).
- [ ] Survival-at-each-layer reported; per-setting survivor counts tabulated.
- [ ] Mapping assumptions written down (scRMSD source, `solubility = camsol_like` heuristic, monomer cutoffs).

**Next:** `04_validate.ipynb` — per-setting trade-offs + the Pareto map.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — per-setting trade-offs + the Pareto map

**Standard slot:** *validate (in silico).* **For Project 02 this is the core science:** for each
setting, summarize recovery vs recapitulation vs diversity vs solubility, then find the
**Pareto-optimal** settings on the foldability ↔ diversity ↔ solubility frontier (D3 part 2). Also
compares consensus-design vs single-sequence `[extension]`.

Needs `results/sequences.csv` with the recapitulation columns from nb 03.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Per-setting metric table

Group the swept sequences by (temperature, noise, seqs) and compute, per cell: mean recovery,
**recapitulation rate** (fraction with scRMSD < 2 Å = foldability), mean per-position entropy
(diversity), and mean solubility proxies. This is the table the cheat-sheet is read off.

In [ ]:
import pandas as pd, numpy as np
from mpnn_tools import shannon_entropy, sequence_recovery

seqs = pd.read_csv("results/sequences.csv")
if "scrmsd" not in seqs:
    raise RuntimeError("Run notebook 03 first to add recapitulation columns.")

# Recovery needs a reference per backbone; in the dry run use the low-temp mock consensus as native.
from mpnn_tools import run_mpnn
ref_by_bb = {bb: run_mpnn(bb, temperature=0.01, noise=0.0, n_seqs=1, tool="mock")[0].sequence
             for bb in seqs["backbone"].unique()}
seqs["recovery"] = [sequence_recovery(ref_by_bb[bb], s)
                    for bb, s in zip(seqs["backbone"], seqs["sequence"])]

def per_position_entropy(group):
    # diversity is per-backbone (sequences for the same backbone are aligned), then averaged
    vals = [shannon_entropy(g["sequence"].tolist())["mean"]
            for _, g in group.groupby("backbone") if len(g) > 1]
    return float(np.mean(vals)) if vals else 0.0

agg = []
for (t, n, s), g in seqs.groupby(["temperature", "noise", "n_seqs"]):
    agg.append(dict(temperature=t, noise=n, n_seqs=s,
                    mean_recovery=g["recovery"].mean(),
                    recap_rate=(g["scrmsd"] < 2.0).mean(),         # foldability
                    mean_entropy=per_position_entropy(g),          # diversity
                    mean_net_charge=g["net_charge"].mean(),
                    mean_hp_frac=g["hydrophobic_fraction"].mean(),
                    mean_camsol_like=g["camsol_like"].mean()))     # solubility proxy
settings = pd.DataFrame(agg)
settings.to_csv("results/settings_summary.csv", index=False)
print("EXAMPLE_DATA (mock backend) — per-setting summary:")
settings.round(3)

## 2 · The Pareto-optimal settings

A setting is **Pareto-optimal** if no other setting beats it on *every* objective at once. We
maximize three objectives: foldability (`recap_rate`), diversity (`mean_entropy`), solubility
(`mean_camsol_like`). The frontier — not a single winner — is the deliverable.

In [ ]:
def pareto_front(df, objectives):
    """Return rows not dominated on all objectives (all maximized)."""
    M = df[objectives].values
    keep = np.ones(len(df), dtype=bool)
    for i in range(len(df)):
        for j in range(len(df)):
            if i != j and np.all(M[j] >= M[i]) and np.any(M[j] > M[i]):
                keep[i] = False
                break
    return df[keep]

OBJ = ["recap_rate", "mean_entropy", "mean_camsol_like"]
front = pareto_front(settings, OBJ).sort_values("recap_rate", ascending=False)
print(f"{len(front)}/{len(settings)} settings are Pareto-optimal on {OBJ}:")
front.round(3)

### Visualize the trade-off

Foldability vs diversity, colored by the solubility proxy. The Pareto front (ringed) is what the
cheat-sheet recommends from, by goal. (Mock data here is illustrative — label any such figure
`EXAMPLE_DATA`.)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4.5))
sc = ax.scatter(settings["recap_rate"], settings["mean_entropy"],
                c=settings["mean_camsol_like"], cmap="viridis", s=60, edgecolor="k", lw=0.3)
ax.scatter(front["recap_rate"], front["mean_entropy"], facecolors="none",
           edgecolors="red", s=180, lw=1.6, label="Pareto-optimal")
for _, row in settings.iterrows():
    ax.annotate(f"T{row.temperature}/N{row.noise}", (row.recap_rate, row.mean_entropy),
                fontsize=6, alpha=0.6)
ax.set_xlabel("recapitulation rate  (foldability)")
ax.set_ylabel("mean per-position entropy  (diversity)")
ax.set_title("MPNN settings trade-off  [EXAMPLE_DATA — mock backend]")
fig.colorbar(sc, label="camsol_like (solubility proxy, NOT real CamSol)")
ax.legend(); plt.tight_layout(); plt.savefig("results/pareto.png", dpi=150); plt.show()

## 3 · Consensus-design vs single-sequence `[extension]`

Does the per-position **consensus** of a backbone's N sequences recapitulate better than the single
best sequence? Consensus often improves stability/expression in the literature — test it here per
backbone and report the win rate (honestly, including ties/losses).

In [ ]:
from collections import Counter
import numpy as np

def consensus(seqs_list):
    L = min(len(s) for s in seqs_list)
    return "".join(Counter(s[i] for s in seqs_list).most_common(1)[0][0] for i in range(L))

# Compare at a single representative setting (e.g., temp 0.2, noise 0.1).
sub = seqs[(seqs.temperature == 0.2) & (seqs.noise == 0.1)]
import hashlib
def mock_recapitulate(s):
    h = int(hashlib.sha256(s.encode()).hexdigest(), 16); return 0.8 + (h % 350)/100.0
wins = 0; total = 0
for bb, g in sub.groupby("backbone"):
    sl = g["sequence"].tolist()
    if len(sl) < 2: continue
    cons_scrmsd = mock_recapitulate(consensus(sl))
    best_single = g["scrmsd"].min()
    wins += int(cons_scrmsd < best_single); total += 1
print(f"consensus beat best-single recapitulation in {wins}/{total} backbones "
      f"[EXAMPLE_DATA — mock]. Report this honestly, including losses, on real data.")

## 4 · ProteinMPNN vs ESM-IF (and FAMPNN if available) `[extension]`

Re-run the sweep (or the best settings) with **ESM-IF** (inverse folding) as a second designer and
compare recovery/recapitulation/diversity head-to-head. Scaffold below — wire in the real backends
on Colab; report per-tool distributions, not single bests.

In [ ]:
# Scaffold (implement on Colab with the real backends):
# for tool in ("proteinmpnn", "esm_if"):           # add "fampnn" if a public release exists
#     pool = sweep_with(tool, BACKBONES, best_settings)
#     recapitulate(pool); summarize(pool)
# compare recovery / recap_rate / entropy across tools (boxplots), report N per tool.
print("Tool-comparison scaffold — ProteinMPNN vs ESM-IF (vs FAMPNN if available). "
      "Verify the current public releases at generation time.")

## D3 (part 2) checklist
- [ ] Per-setting table: recovery, recapitulation rate, entropy, solubility proxies (`results/settings_summary.csv`).
- [ ] Pareto-optimal settings identified + the trade-off figure (foldability ↔ diversity ↔ solubility).
- [ ] Consensus-vs-single win rate reported honestly (incl. losses).
- [ ] ProteinMPNN vs ESM-IF comparison (per-tool distributions, N per tool).
- [ ] Honest per-setting hit rate (N recapitulating / N generated). No single "best" claimed.

**Next:** `05_validation_plan.ipynb` — the cheat-sheet + the wet-lab plan.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Cheat-sheet + Expression/Validation Plan

**Standard slot:** *validation plan.* **For Project 02 this means:** turn the Pareto map (nb 04) into
the **MPNN settings cheat-sheet** (D★) and a concrete expression + validation plan with controls
(D4/D5). Also the recommendation heuristic and the MPNNsol surface-redesign stretch task.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · The MPNN settings cheat-sheet (D★)

Read the recommendations off your Pareto front (nb 04), by goal. Fill the `?`s from
`results/settings_summary.csv` — this card is the deliverable later cohort projects actually use.
Save it to the repo root / `data/CHEATSHEET.md`.

In [ ]:
cheatsheet = """# MPNN Settings Cheat-Sheet (Project 02 — by <your name>, <date>)

Read off the Pareto front in notebook 04. NO setting is universally best; pick by goal.
Numbers below are PLACEHOLDERS — fill from results/settings_summary.csv (and label the dataset).

| Goal | Recommended temp | Recommended noise | seqs/backbone | Why (from your Pareto map) |
|------|------------------|-------------------|---------------|----------------------------|
| Max foldability (safe, conservative)   | ~0.1 | 0.0      | 8–16 | highest recapitulation rate; low diversity |
| Balanced (default for most pipelines)  | ~0.2 | 0.0–0.1  | 16   | good recap rate, usable diversity, OK solubility |
| Max diversity (give the filter choices)| ~0.3–0.5 | 0.1–0.2 | 48 | high entropy; expect lower recap rate — over-generate then filter |
| Max predicted solubility               | ~0.1–0.2 | 0.0    | 16–48 | best camsol_like proxy + low hydrophobic-patch fraction |

## Honest caveats (state these every time)
- Recapitulation (scRMSD < 2 A) is self-consistency, NOT proof of folding/stability/expression.
- net_charge / hydrophobic_fraction / camsol_like are PROXIES. camsol_like is a heuristic, NOT real
  CamSol (Sormanni 2015). Only express -> SDS-PAGE -> SEC measures expression.
- Cutoffs are calibrated on <your backbone set>; revisit for different fold classes / lengths.
- "diversity before filtering": prefer over-generating at higher temp and filtering hard over
  polishing one low-temp sequence.
"""
open("../data/CHEATSHEET.md", "w").write(cheatsheet)
print("wrote ../data/CHEATSHEET.md — fill the placeholders from notebook 04.")

## 2 · Settings-recommendation tool `[extension]`

A tiny heuristic that, given a backbone descriptor and a goal, returns a recommended setting from
your Pareto front. Wire it to `results/settings_summary.csv` for the real version; this stub encodes
the cheat-sheet logic so downstream students can import it.

In [ ]:
def recommend_settings(goal="balanced", length=120, ss_class="mixed"):
    """Recommend (temperature, noise, n_seqs) for a goal. Replace the table with your Pareto front.

    goal in {"foldability", "balanced", "diversity", "solubility"}.
    On the real version, load results/settings_summary.csv and pick the Pareto-optimal cell that
    maximizes the chosen objective subject to a minimum recapitulation rate.
    """
    table = {
        "foldability": dict(temperature=0.1, noise=0.0, n_seqs=16),
        "balanced":    dict(temperature=0.2, noise=0.1, n_seqs=16),
        "diversity":   dict(temperature=0.5, noise=0.2, n_seqs=48),
        "solubility":  dict(temperature=0.1, noise=0.0, n_seqs=48),
    }
    if goal not in table:
        raise ValueError(f"goal must be one of {sorted(table)}")
    rec = dict(table[goal]); rec["goal"] = goal
    return rec

for g in ("foldability", "balanced", "diversity", "solubility"):
    print(g, "->", recommend_settings(goal=g))

## 3 · Expression strategy + validation plan (D4)

The plan a wet lab would follow to test your recommended settings. **Controls are mandatory.**

In [ ]:
plan = """# Expression + Validation Plan (Project 02)

## Codon optimization & construct
- Optimize chosen sequences for the host's codon usage (e.g., E. coli); avoid rare-codon clusters
  and strong mRNA secondary structure near the start codon.
- N-terminal His6 tag with a cleavable linker (TEV) for IMAC purification; keep the tag out of any
  predicted hydrophobic patch.
- Order genes ONLY through a biosecurity-screening provider (IGSC member).

## Expression
- Host: E. coli BL21(DE3). Induce at 16-18 C overnight (low temperature favors soluble de novo
  monomer expression). Note when a different host is needed.

## Characterization tiers
1. Go/no-go: express -> SDS-PAGE (soluble vs inclusion bodies) -> SEC (monodisperse? right size?).
2. Basic: DSF (Tm) and/or CD (secondary structure matches the design?).
3. Deep (top picks): structure (X-ray/cryo-EM) or SEC-MALS/SAXS.

## Controls (MANDATORY)
- Positive control: a KNOWN-GOOD NATURAL monomer sequence (expresses + folds well).
- Negative control: a DELIBERATELY HIGH-HYDROPHOBIC-PATCH design (expected to express poorly /
  aggregate) — tests that the solubility proxy points the right way.
- Unrelated-protein control.

## Reporting
- Report the EXPRESSION HIT RATE per setting (N soluble / N tried), not the cherry. Include failures.
- Never present in-silico proxy scores as expression results.
"""
open("../data/PLAN.md", "w").write(plan)
print("wrote ../data/PLAN.md")

## 4 · (Stretch) MPNNsol / surface-redesign comparison `[stretch]`

Redesign **only the surface positions** for solubility (an MPNNsol-style pass that fixes the core and
lets surface residues vary toward soluble identities), then re-score: did the solubility proxies
improve without hurting recapitulation? Scaffold below — implement with real MPNN fixed-position
masks on Colab.

In [ ]:
# Scaffold (implement on Colab):
# 1. Compute per-residue burial (SASA from the backbone) -> classify core vs surface.
# 2. Re-run ProteinMPNN with --fixed_positions_jsonl pinning the CORE; let SURFACE vary.
# 3. Re-recapitulate + re-score net_charge / hydrophobic_fraction / camsol_like.
# 4. Compare surface-redesigned vs original: solubility proxy delta vs recapitulation delta.
print("MPNNsol surface-redesign scaffold — fix core, vary surface, re-score solubility proxies.")

## D4 / D5 checklist
- [ ] `data/CHEATSHEET.md` completed from the Pareto front, by goal, with honest caveats (D★).
- [ ] `recommend_settings(...)` wired to `results/settings_summary.csv`.
- [ ] `data/PLAN.md`: codon/tag + express→SDS-PAGE→SEC→DSF/CD + the three controls (incl. the
      high-hydrophobic-patch negative) + timeline + costed reagents.
- [ ] (Stretch) MPNNsol surface-redesign comparison.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — and the cohort's MPNN-running projects now have your settings cheat-sheet.